In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json

import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,3,4,5,6,7"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils
import torch
import transformers
from rich.console import Console
from rich.theme import Theme
import builtins
_orig_print = builtins.print

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme, width = 300)


# Override print globally
def print(*args, **kwargs):
    console.print(*args, **kwargs)


logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

/disk/u/arnab/miniconda3/envs/goals/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-02-19 12:16:42 __main__ INFO     torch.__version__='2.10.0+cu128', torch.version.cuda='12.8'
2026-02-19 12:16:42 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=7, torch.cuda.get_device_name()='NVIDIA A100 80GB PCIe'
2026-02-19 12:16:42 __main__ INFO     transformers.__version__='5.1.0'


## Loading the LM

In [3]:
from src.models import ModelandTokenizer

# model_key = "allenai/Olmo-3-1125-32B"
# model_key = "allenai/Olmo-3.1-32B-Instruct"
# model_key = "allenai/Olmo-3.1-7B-RL-Zero-Code"

# model_key = "meta-llama/Llama-3.3-70B-Instruct"

# model_key = "deepseek-ai/deepseek-coder-33b-base"
# model_key = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"

# model_key = "openai/gpt-oss-20b"
# model_key = "openai/gpt-oss-120b" #! weird results. known issue.

# model_key = "Qwen/Qwen3-Coder-30B-A3B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-14B"
# model_key = "Qwen/Qwen2.5-Coder-14B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-32B"
model_key = "Qwen/Qwen2.5-Coder-32B-Instruct"
# model_key = "Qwen/Qwen2.5-72B"
# model_key = "Qwen/Qwen2.5-72B-Instruct"

# model_key = "bigcode/starcoder2-15b"

# model_key = "zai-org/GLM-4.7-Flash"

mt = ModelandTokenizer(
    model_key=model_key,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

2026-02-19 12:16:46 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-02-19 12:16:46 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-02-19 12:16:46 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f5cb325a060>
2026-02-19 12:16:46 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f5cb45a41d0> server_hostname='huggingface.co' timeout=10
2026-02-19 12:16:46 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f5cb38ae270>
2026-02-19 12:16:46 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-02-19 12:16:46 httpcore.http11 DEBUG    send_request_headers.complete
2026-02-19 12:16:46 http

Loading weights: 100%|██████████| 771/771 [00:17<00:00, 43.24it/s, Materializing param=model.norm.weight]                              


2026-02-19 12:17:10 httpcore.connection DEBUG    close.started
2026-02-19 12:17:10 httpcore.connection DEBUG    close.complete
2026-02-19 12:17:10 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-02-19 12:17:10 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f5caf9a5610>
2026-02-19 12:17:10 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f5cb45a41d0> server_hostname='huggingface.co' timeout=10
2026-02-19 12:17:10 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f5caf9a5ca0>
2026-02-19 12:17:10 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-02-19 12:17:10 httpcore.http11 DEBUG    send_request_headers.complete
2026-02-19 12:17:10 httpcore.http11 DEBUG    send_request_body.started request=<Request [b'HE

In [4]:
from src.goals.eval import run_evaluation, pretty_print_eval_results
from src.goals.data import IncompleteCodeDataset


SAMPLE_IDX = 5

eval_dataset = IncompleteCodeDataset.load_dataset()
print(eval_dataset[SAMPLE_IDX])

2026-02-19 12:17:10 src.goals.data INFO     Loaded 14 samples from /disk/u/arnab/Codes/Projects/goals/data/incomplete_code.json


IncompleteCodeSample(
    instruction='Write a function in python that checks if a string is a palindrome.\n\n',
    code='def is_palindrome(s: str) -> bool:\n    return True if ',
    unit_tests=[('racecar', True), ('hello', False), ('', True), ('a', True), ('ab', False), ('aba', True), ('abba', True)],
    prefills={'aligned': ['', '(', 's', 's ==', 's == s', 's[::', 's[::-1]', '(s', 'list(', 'list(s)'], 'adversarial': ['not', 'not (', 's +', '3', '1 +'], 'inconsequential': ['bool(', 'True and', 'False or', '0 +']},
    function_name='is_palindrome'
)

In [5]:
from src.tokens import prepare_input
from src.goals.data import CodeGenerations, Code
from src.goals.functional import supports_chat_template, apply_chat_template
from src.functional import generate_with_patch

sample = eval_dataset[SAMPLE_IDX]
instruction = sample.instruction
unfinished_code = sample.code
function_name = sample.function_name or "func"
unit_tests = sample.unit_tests
prefills = []
for ptype, pfill_list in sample.prefills.items():
    prefills.extend([prefill, ptype] for prefill in pfill_list)


if supports_chat_template(mt):
    prompts = [
        apply_chat_template(
            mt=mt,
            user_query=instruction.strip(),
            incomplete_code=unfinished_code + prefill,
        )
        for prefill, ptype in prefills
    ]
else:
    prompts = [
        instruction + "<code>" + unfinished_code + prefill
        for (prefill, ptype) in prefills
    ]


inputs = prepare_input(
    prompts=prompts,
    tokenizer=mt,
)

output = generate_with_patch(
    mt=mt,
    inputs=inputs,
    n_gen_per_prompt=1,
    max_new_tokens=64,
    do_sample=False,
    remove_prefix=True,
)

results = CodeGenerations(
    sample=sample,
    generations=[
        Code(prefill=prefill, prefill_type=ptype, full_prompt=inp, lm_generation=out)
        for (prefill, ptype), inp, out in zip(prefills, prompts, output)
    ],
)

# print(f"[white]{str(results)}[/white]")

eval_results = run_evaluation(
    result = results,
    function_name=function_name,
    code_stop_tag="</code>",
)

# Print results in tabular format
pretty_print_eval_results(eval_results)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


True
False
True
False
True
False
True
False
True
False
True
False
True
False
True
False
True
False
True
False
True
False
True
False
True
False
True
False
True
False
True
False


"""def is_palindrome(s: str) -> bool:
    return True if """

                                                                                Code Completion Results                                                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Prefill                     ┃ Logic                                                          ┃ Verdict ┃ Unit Tests ┃ Logs                                                          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ aligned: ""                 │ " s == s[::-1] else False                                      │ PASSED  │            │                                                               │
│                             │                                                                │         │            │                                                               │
│                             │ # Example usage:                                               │         │            │                                                               │
│                             │ print(is_palindrome("racecar"))  # Output: True                │         │            │                                                               │
│                             │ print(is_palindrome("hello"))    # Output: False"              │         │            │                                                               │
├─────────────────────────────┼────────────────────────────────────────────────────────────────┼─────────┼────────────┼───────────────────────────────────────────────────────────────┤
│ aligned: "("                │ "(s == s[::-1]) else False                                     │ PASSED  │            │                                                               │
│                             │                                                                │         │            │                                                               │
│                             │ # Example usage:                                               │         │            │                                                               │
│                             │ print(is_palindrome("racecar"))  # Output: True                │         │            │                                                               │
│                             │ print(is_palindrome("hello"))    # Output: False"              │         │            │                                                               │
├─────────────────────────────┼────────────────────────────────────────────────────────────────┼─────────┼────────────┼───────────────────────────────────────────────────────────────┤
│ aligned: "s"                │ "s == s[::-1] else False                                       │ PASSED  │            │                                                               │
│                             │                                                                │         │            │                                                               │
│                             │ # Example usage:                                               │         │            │                                                               │
│                             │ print(is_palindrome("racecar"))  # Output: True                │         │            │                                                               │
│                             │ print(is_palindrome("hello"))    # Output: False"              │         │            │                                                               │
├─────────────────────────────┼────────────────────────────────────────────────────────────────┼─────────┼────────────┼─────────────────

Summary: 17/19 passed, 2/19 failed

In [42]:
prefill_idx = -5
generated_code = eval_results.generations[prefill_idx]
print(generated_code)

print(f'Full Prompt:\n[white]"{generated_code.full_prompt}"[/white]\n')
print(f'Full Output:\n[white]"{generated_code.lm_generation}"[/white]\n')

Code(
    prefill='1 +',
    prefill_type='adversarial',
    full_prompt='<|im_start|>system\nYou are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>\n<|im_start|>user\nWrite a function in python that checks if a string 
is a palindrome.<|im_end|>\n<|im_start|>assistant\n<code>\ndef is_palindrome(s: str) -> bool:\n    return True if 1 +',
    lm_generation=' s == s[::-1] else False\n\n# Example usage:\nprint(is_palindrome("racecar"))  # Output: True\nprint(is_palindrome("hello"))    # Output: False',
    source_code='def is_palindrome(s: str) -> bool:\n    return True if 1 + s == s[::-1] else False\n\n# Example usage:\nprint(is_palindrome("racecar"))  # Output: True\nprint(is_palindrome("hello"))    # Output: False',
    function_name='is_palindrome',
    eval=SingleCodeEval(
        code='def is_palindrome(s: str) -> bool:\n    return True if 1 + s == s[::-1] else False\n\n# Example usage:\nprint(is_palindrome("racecar"))  # Output: True\nprint(is_palindrome("hello"))    # Output: False',
        verdict='ERROR',
        unit_test_results=[],
        warnings=[],
        error_message="TypeError: unsupported operand type(s) for +: 'int' and 'str'",
        prefill=('1 +', 'adversarial'),
        output=' s == s[::-1] else False\n\n# Example usage:\nprint(is_palindrome("racecar"))  # Output: True\nprint(is_palindrome("hello"))    # Output: False'
    )
)

Full Prompt:
"<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a function in python that checks if a string is a palindrome.<|im_end|>
<|im_start|>assistant
<code>
def is_palindrome(s: str) -> bool:
    return True if 1 +"

Full Output:
" s == s[::-1] else False

# Example usage:
print(is_palindrome("racecar"))  # Output: True
print(is_palindrome("hello"))    # Output: False"

In [43]:
from src.goals.sampling import MCTSSearch

full_prompt = prompts[prefill_idx]
print(f'Full Prompt for MCTS:\n[white]"{full_prompt}"[/white]\n')

search = MCTSSearch(
    mt=mt,
    prompt=full_prompt,
    max_depth=32, top_k=5,
    prob_threshold=0.01,
    temperature=1.0,
    c_puct=1.0,
    min_children_up_to_depth=3,
    min_children=2,
)

root = search.run(
    n_iterations=1500,
    depth_first=True,
)

Full Prompt for MCTS:
"<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a function in python that checks if a string is a palindrome.<|im_end|>
<|im_start|>assistant
<code>
def is_palindrome(s: str) -> bool:
    return True if 1 +"

2026-02-19 13:06:04 src.goals.sampling DEBUG    MCTS iteration 50/1500, total nodes: 908
2026-02-19 13:06:19 src.goals.sampling DEBUG    MCTS iteration 100/1500, total nodes: 1197
2026-02-19 13:06:30 src.goals.sampling DEBUG    MCTS iteration 150/1500, total nodes: 1426
2026-02-19 13:06:36 src.goals.sampling DEBUG    MCTS iteration 200/1500, total nodes: 1522
2026-02-19 13:06:47 src.goals.sampling DEBUG    MCTS iteration 250/1500, total nodes: 1730
2026-02-19 13:06:51 src.goals.sampling DEBUG    MCTS iteration 300/1500, total nodes: 1793
2026-02-19 13:06:59 src.goals.sampling DEBUG    MCTS iteration 350/1500, total nodes: 1945
2026-02-19 13:07:04 src.goals.sampling DEBUG    MCTS iteration 400/1500, total nodes: 2064
2026-02-19 13:07:14 src.goals.sampling DEBUG    MCTS iteration 450/1500, total nodes: 2255
2026-02-19 13:07:26 src.goals.sampling DEBUG    MCTS iteration 500/1500, total nodes: 2463
2026-02-19 13:07:38 src.goals.sampling DEBUG    MCTS iteration 550/1500, total nodes: 2675
2

In [57]:
print(search.get_stats())

paths = search.get_all_paths(
    # terminal_only=True, 
    sort_by="prob", 
    # contains="else False"
    # depth=5
)

# Print results
for p in paths:
    _orig_print(
        f'<v={p["visits"]}, p={p["prob"]:.4f}> {repr(p["text"])}'
    )

{'total_iterations': 1500, 'total_nodes': 4277, 'terminal_count': 354, 'max_depth_reached': 32, 'avg_depth': 18.115819209039547, 'root_visits': 3599, 'terminal_reasons': {'stop_string:\n': 211, 'stop_token': 109, 'max_depth': 34}}

<v=180, p=0.0490> ' s == s[::-1] else False\n\n'
<v=149, p=0.0412> ' (s == s[::-1]) else False\n\n'
<v=95, p=0.0287> ' s[::-1] == s else False\n\n'
<v=77, p=0.0254> ' s.find(s[::-1]) else False\n\n'
<v=84, p=0.0231> ' s == s[::-1] else False\n'
<v=42, p=0.0206> ' len(s) <= 2 or s == s[::-1] else False\n\n'
<v=37, p=0.0181> ' len(s) <= 2 else s[0] == s[-1] and is_palindrome(s[1:-1])\n\n'
<v=37, p=0.0165> ' s[::1] == s[::-1] else False\n\n'
<v=51, p=0.0161> ' (s != s[::-1]) == 1 else False\n\n'
<v=39, p=0.0146> ' (s[::-1] == s) else False\n\n'
<v=29, p=0.0136> ' 2 == 3 else s == s[::-1]\n'
<v=38, p=0.0134> ' s == s + 1 else s == s[::-1]\n'
<v=39, p=0.0122> ' 1 == 2 else s == s[::-1]\n'
<v=35, p=0.0108> ' 1 == 2 else s == s[::-1]\n\n'
<v=35, p=0.0106> ' s[::-1] == s else False\n'
<v=23, p=0.0106> ' 2 == 3 else s == s[::-1]\n\n'
<v=20, p=0.0097> ' len(s) <= 2 or s == s[::-1] else False\n'
<v=28, p=0.0094> ' s.find(s[::-1]) else False\n'
<v=33, p=0.0092> ' (s == s[::-1]) else False\n'
<v=26

In [58]:
# Inspect results
search.print_tree(max_depth=None, show_colors=False)

[ROOT] visits=3599
    ├── ' s' (p=0.371, v=1353)
    │   ├── ' ==' (p=0.338, v=462)
    │   │   ├── ' s' (p=0.953, v=436)
    │   │   │   ├── '[::-' (p=0.613, v=269)
    │   │   │   │   └── '1' (p=1.000, v=268)
    │   │   │   │       └── ']' (p=1.000, v=267)
    │   │   │   │           └── ' else' (p=0.988, v=266)
    │   │   │   │               └── ' False' (p=1.000, v=265)
    │   │   │   │                   ├── '\n\n' (p=0.676, v=180) [TERMINAL] (' s == s[::-1] else False\n\n')
    │   │   │   │                   └── '\n' (p=0.318, v=84) [TERMINAL] (' s == s[::-1] else False\n')
    │   │   │   ├── ' +' (p=0.256, v=112)
    │   │   │   │   └── ' ' (p=0.992, v=112)
    │   │   │   │       └── '1' (p=1.000, v=111)
    │   │   │   │           ├── ' else' (p=0.961, v=101)
    │   │   │   │           │   ├── ' s' (p=0.949, v=88)
    │   │   │   │           │   │   ├── ' ==' (p=0.820, v=68)
    │   │   │   │           │   │   │   └── ' s' (p=1.000, v=67)
    │   │   │   │           │   

In [59]:
from src.functional import predict_next_token

prefix = sample.code + generated_code.prefill
print(prefix)

test = ' == n % 2)'

test_prompt = generated_code.full_prompt + test
_orig_print(test_prompt)

predict_next_token(
    mt=mt,
    inputs = test_prompt
)

def is_palindrome(s: str) -> bool:
    return True if 1 +

<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a function in python that checks if a string is a palindrome.<|im_end|>
<|im_start|>assistant
<code>
def is_palindrome(s: str) -> bool:
    return True if 1 + == n % 2)


[[PredictedToken(token=' {\n', prob=0.25390625, logit=16.875, token_id=341, metadata=None),
  PredictedToken(token=' else', prob=0.197265625, logit=16.625, token_id=770, metadata=None),
  PredictedToken(token=' and', prob=0.11962890625, logit=16.125, token_id=323, metadata=None),
  PredictedToken(token=' *', prob=0.056640625, logit=15.375, token_id=353, metadata=None),
  PredictedToken(token=' or', prob=0.050048828125, logit=15.25, token_id=476, metadata=None)]]

In [60]:
from src.goals.eval import evaluate_mcts_paths, pretty_print_mcts_eval

evaluated_paths = evaluate_mcts_paths(
    prefix = prefix,
    paths = paths,
    unit_tests= sample.unit_tests,
    function_name=sample.function_name or "func",
)

# pretty_print_mcts_eval(
#     prefix = prefix,
#     paths = evaluated_paths,
#     function_name=sample.function_name or "func",
# )

In [61]:
# print(search.get_stop_tokens_info())

In [62]:
# node = search.find_node_by_text(' == n % 2)')
# node

In [63]:
# # Debug why it's not being expanded
# if node:
#     print(search.debug_node(node))

#     # If it's not terminal, see what children would be created
#     if not node.is_terminal and node.is_leaf():
#         search.force_expand(node)

#     # Check PUCT scores at its parent to see why siblings might be selected first
#     if node.parent:
#         for child, score in search.get_puct_scores(node.parent):
#             print(f"{child.token_str!r}: PUCT={score:.4f}, visits={child.visits}")

In [64]:
from src.goals.eval import evaluate_mcts_tree, clear_mcts_colors

prefix = sample.code + generated_code.prefill
clear_mcts_colors(search.root)

counts = evaluate_mcts_tree(
    root=search.root,
    prefix=prefix,
    unit_tests=sample.unit_tests,
    function_name=sample.function_name or "func",
    terminal_only=True
)

print(counts)

{'green': 104, 'orange': 228, 'red': 22, 'total': 354}

In [65]:
search.print_tree(max_depth=None, show_colors=True)

[ROOT] visits=3599
    ├── ' s' (p=0.371, v=1353)
    │   ├── ' ==' (p=0.338, v=462)
    │   │   ├── ' s' (p=0.953, v=436)
    │   │   │   ├── '[::-' (p=0.613, v=269)
    │   │   │   │   └── '1' (p=1.000, v=268)
    │   │   │   │       └── ']' (p=1.000, v=267)
    │   │   │   │           └── ' else' (p=0.988, v=266)
    │   │   │   │               └── ' False' (p=1.000, v=265)
    │   │   │   │                   ├── '\n\n' (p=0.676, v=180) [TERMINAL] (' s == s[::-1] else False\n\n')
    │   │   │   │                   └── '\n' (p=0.318, v=84) [TERMINAL] (' s == s[::-1] else False\n')
    │   │   │   ├── ' +' (p=0.256, v=112)
    │   │   │   │   └── ' ' (p=0.992, v=112)
    │   │   │   │       └── '1' (p=1.000, v=111)
    │   │   │   │           ├── ' else' (p=0.961, v=101)
    │   │   │   │           │   ├── ' s' (p=0.949, v=88)
    │   │   │   │           │   │   ├── ' ==' (p=0.820, v=68)
    │   │   │   │           │   │   │   └── ' s' (p=1.000, v=67)
    │   │   │   │           │   │   │       └── '[::-' (p=1.000, v=66)
    │   │   │   │           │   │   │           └── '1' (p=1.000, v=65)
    │   │   │   │           │   │   │               ├── ']\n' (p=0.590, v=38) [TERMINAL] (' s == s + 1 else s == s[::-1]\n')
    │   │   │   │           │   │   │               └── ']\n\n' (p=0.406, v=26) [TERMINAL] (' s == s + 1 else s == s[::-1]\n\n')
    │   │   │   │           │   │   ├── '[::-' (p=0.143, v=11)
    │   │   │   │           │   │   │   └── '1' (p=1.000, v=11)
    │   │   │   │           │   │   │       ├── ']' (p=0.953, v=10)
    │   │   │   │           │   │   │       │   └── ' ==' (p=1.000, v=9)
    │   │   │   │           │   │   │       │       └── ' s' (p=1.000, v=8)
    │   │   │   │           │   │   │       │           ├── '\n\n' (p=0.699, v=5) [TERMINAL] (' s == s + 1 else s[::-1] == s\n\n')
    │   │   │   │           │   │   │       │           └── '\n' (p=0.293, v=2) [TERMINAL] (' s == s + 1 else s[::-1] == s\n')
    │   │   │   │           │   │   │       └── '].' (p=0.047, v=0) (' s == s + 1 else s[::-1].')
    │   │   │   │           │   │   └── '.lower' (p=0.032, v=8)
    │   │   │   │           │   │       ├── '()' (p=0.902, v=8)
    │   │   │   │           │   │       │   └── ' ==' (p=1.000, v=7)
    │   │   │   │           │   │       │       └── ' s' (p=1.000, v=6)
    │   │   │   │           │   │       │           ├── '[::-' (p=0.961, v=5)
    │   │   │   │           │   │       │           │   └── '1' (p=1.000, v=4)
    │   │   │   │           │   │       │           │       └── '].' (p=1.000, v=3)
    │   │   │   │           │   │       │           │           └── 'lower' (p=1.000, v=2)
    │   │   │   │           │   │       │           │               ├── '()\n\n' (p=0.797, v=1) [TERMINAL] (' s == s + 1 else s.lower() == s[::-1].lower()\n\n')
    │   │   │   │           │   │       │           │               └── '()\n' (p=0.201, v=0) [TERMINAL] (' s == s + 1 else s.lower() == s[::-1].lower()\n')
    │   │   │   │           │   │       │           └── '.lower' (p=0.037, v=0) (' s == s + 1 else s.lower() == s.lower')
    │   │   │   │           │   │       ├── '().' (p=0.084, v=0) (' s == s + 1 else s.lower().')
    │   │   │   │           │   │       └── '()[' (p=0.015, v=0) (' s == s + 1 else s.lower()[')
    │   │   │   │           │   └── ' False' (p=0.047, v=12)
    │   │   │   │           │       ├── '\n' (p=0.418, v=2) [TERMINAL] (' s == s + 1 else False\n')
    │   │   │   │           │       ├── '\n\n' (p=0.224, v=1) [TERMINAL] (' s == s + 1 else False\n\n')
    │   │   │   │           │       ├── ' if' (p=0.197, v=9)
    │   │   │   │           │       │   ├── ' s' (p=0.664, v=9)
    │   │   │   │           │       │   │   ├── ' !=' (p=0.543, v=8)
    │   │   │   │           │       │   │   │   └── ' s' (p=0.996, v=7)
    │   │   │   │           │       │   │   │       └── '[::-' (p=1.000, v=6)
    │   │   │   │           │       │   │   │           └── '1' (p=1.000, v=5)
    │   │   

In [66]:
search.save_tree("tree.html", show_colors=True)

2026-02-19 13:11:53 src.goals.sampling INFO     Saved tree to tree.html (HTML with colors)


## Checking contrasting sampling

In [67]:
from src.goals.probe import (
    compute_outcome_probs,
    sample_contrastive_pairs,
    sample_multidepth_pairs,
    print_contrastive_pairs,
    summarize_contrastive_pairs,
)

# Sample contrastive pairs from the evaluated MCTS tree
# The tree should already be evaluated (search.root has colors assigned)
pairs_by_depth = sample_multidepth_pairs(
    root=search.root,
    min_depth=1,
    max_depth=16,
    min_green_confidence=0.5,
    max_red_in_positive=0.3,  # Slightly relaxed
    min_red_confidence=0.5,
    max_green_in_negative=0.1,
    max_uncertain=0.5,
)

print_contrastive_pairs(pairs_by_depth, max_pairs_per_depth=5)

2026-02-19 13:11:54 src.goals.probe DEBUG    depth=1: 4 nodes, 0 positive, 0 negative
2026-02-19 13:11:54 src.goals.probe DEBUG    depth=2: 12 nodes, 1 positive, 4 negative
2026-02-19 13:11:54 src.goals.probe DEBUG    depth=3: 38 nodes, 3 positive, 9 negative
2026-02-19 13:11:54 src.goals.probe DEBUG    depth=4: 93 nodes, 6 positive, 16 negative
2026-02-19 13:11:54 src.goals.probe DEBUG    depth=5: 118 nodes, 13 positive, 24 negative
2026-02-19 13:11:54 src.goals.probe DEBUG    depth=6: 193 nodes, 20 positive, 33 negative


2026-02-19 13:11:54 src.goals.probe DEBUG    depth=7: 234 nodes, 31 positive, 41 negative
2026-02-19 13:11:54 src.goals.probe DEBUG    depth=8: 294 nodes, 36 positive, 54 negative
2026-02-19 13:11:54 src.goals.probe DEBUG    depth=9: 375 nodes, 44 positive, 54 negative
2026-02-19 13:11:54 src.goals.probe DEBUG    depth=10: 316 nodes, 45 positive, 56 negative
2026-02-19 13:11:54 src.goals.probe DEBUG    depth=11: 285 nodes, 44 positive, 58 negative
2026-02-19 13:11:54 src.goals.probe DEBUG    depth=12: 273 nodes, 39 positive, 53 negative
2026-02-19 13:11:54 src.goals.probe DEBUG    depth=13: 262 nodes, 33 positive, 46 negative
2026-02-19 13:11:54 src.goals.probe DEBUG    depth=14: 210 nodes, 29 positive, 40 negative
2026-02-19 13:11:54 src.goals.probe DEBUG    depth=15: 195 nodes, 27 positive, 38 negative
2026-02-19 13:11:54 src.goals.probe DEBUG    depth=16: 147 nodes, 21 positive, 36 negative


Contrastive Pairs Summary

Total pairs: 18289

Depths with pairs: 15

Depth range: (2, 16)

... (42 outputs left out)

... and 1021 more pairs

Depth 16 (756 pairs)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Positive Path                     ┃ P(green) ┃ Negative Path                                  ┃ P(red) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ ' len(s) <= 2 or s\[0] == s\[-1]' │     0.99 │ ' s == s + 1 else s.lower() == s\[::-1].lower' │   1.00 │
│ ' len(s) <= 2 or s\[0] == s\[-1]' │     0.99 │ ' s == s + 1 else False if s != s\[::-1] else' │   1.00 │
│ ' len(s) <= 2 or s\[0] == s\[-1]' │     0.99 │ ' s\[::-1] == s + 1 else s\[::-1] == s'        │   1.00 │
│ ' len(s) <= 2 or s\[0] == s\[-1]' │     0.99 │ " s\[::-1] == s + '1' else s\[::-1] =="        │   1.00 │
│ ' len(s) <= 2 or s\[0] == s\[-1]' │     0.99 │ ' s\[::-1] == s\[::-1] + 1 else s == s'        │   1.00 │
└───────────────────────────────────┴──────────┴────────────────────────────────────────────────┴────────┘

... and 751 more pairs

In [68]:
# Inspect outcome probabilities at a specific depth
depth_to_inspect = 3
from src.goals.probe import collect_nodes_at_depth

nodes = collect_nodes_at_depth(search.root, depth_to_inspect)
print(f"Nodes at depth {depth_to_inspect}: {len(nodes)}")

for node in nodes[:10]:  # Show first 10
    if not node.is_leaf() and not node.is_terminal:
        probs = compute_outcome_probs(node)
        path_text = node.get_path_text()[:40]
        print(f"  '{path_text}...' -> {probs}")

Nodes at depth 3: 38

' s == s...' -> OutcomeProbs(green=0.000, red=0.950, uncertain=0.050)

' s == (...' -> OutcomeProbs(green=0.000, red=0.537, uncertain=0.463)

' s == ...' -> OutcomeProbs(green=0.000, red=0.906, uncertain=0.094)

' s[::-1...' -> OutcomeProbs(green=0.123, red=0.831, uncertain=0.046)

' s[::-2...' -> OutcomeProbs(green=0.000, red=0.156, uncertain=0.844)

' s.find(s...' -> OutcomeProbs(green=0.789, red=0.179, uncertain=0.032)

' s.find('...' -> OutcomeProbs(green=0.000, red=0.030, uncertain=0.970)

' s[:len...' -> OutcomeProbs(green=0.000, red=0.356, uncertain=0.644)

' s[:1...' -> OutcomeProbs(green=0.000, red=0.066, uncertain=0.934)

' s[:int...' -> OutcomeProbs(green=0.000, red=0.042, uncertain=0.958)

In [ ]:
# Look at a specific contrastive pair in detail
if pairs_by_depth:
    # Get pairs from the first depth that has them
    first_depth = min(pairs_by_depth.keys())
    pairs = pairs_by_depth[first_depth]
    
    if pairs:
        pair = pairs[0]
        print(f"[bold]Example Contrastive Pair at depth {first_depth}:[/bold]")
        print()
        print(f"[green]Positive path:[/green] {repr(pair.positive.get_path_text())}")
        print(f"  Outcome probs: {pair.pos_probs}")
        print(f"  Node has {len(pair.positive.children)} children")
        print()
        print(f"[red]Negative path:[/red] {repr(pair.negative.get_path_text())}")
        print(f"  Outcome probs: {pair.neg_probs}")
        print(f"  Node has {len(pair.negative.children)} children")
else:
    print("[yellow]No contrastive pairs found. Try adjusting thresholds or running more MCTS iterations.[/yellow]")

Example Contrastive Pair at depth 2:

Positive path: ' s.find'

Outcome probs: OutcomeProbs(green=0.785, red=0.179, uncertain=0.036)

Node has 2 children

Negative path: ' s =='

Outcome probs: OutcomeProbs(green=0.000, red=0.938, uncertain=0.062)

Node has 3 children

: 